# 📡 NPU-on-PYNQ — 16×16 Weight-Stationary Systolic Array

**SiliconToAI Project** | KAIST EE | PYNQ-Z2 (xc7z020)

This notebook controls a custom 16×16 weight-stationary NPU implemented in PL fabric
via AXI-Lite MMIO from the Zynq PS ARM core.

### Architecture
```
  ARM Cortex-A9 (PS)
       │ M_AXI_GP0
       ▼
  AXI Interconnect
       │ AXI4-Lite @ 0x43C0_0000
       ▼
  ┌───────────────────────────────┐
  │  fpga_pynq_top (PL)         │
  │                               │
  │  MMIO Regs ──▶ npu_system_top │
  │                │              │
  │  Local BRAM ◀─┘ DMA Engine   │
  │  (test data)   │              │
  │                ▼              │
  │  16×16 Systolic Array         │
  │  (weight-stationary MAC)     │
  │                │              │
  │  Accumulator SRAM            │
  │                │              │
  │  Drain ─▶ PSUM Readback Regs  │
  └───────────────────────────────┘
```

## 1. Load PYNQ Overlay

In [ ]:
from pynq import Overlay, MMIO
import numpy as np
import matplotlib.pyplot as plt
import time

# Load the NPU overlay (requires npu_overlay.bit + npu_overlay.hwh in same dir)
ol = Overlay("npu_overlay.bit")
print(f"Overlay loaded: {ol.bitfile_name}")
print(f"IP blocks: {ol.ip_dict.keys()}")

## 2. Register Map & NPU Driver Class

In [ ]:
class NPUDriver:
    """MMIO driver for the 16x16 weight-stationary NPU."""

    # Register offsets
    CTRL      = 0x00   # W1S: [0]start_dma [1]start_npu [2]swap_banks [3]clear_done
    STATUS    = 0x04   # RO:  [0]dma_done  [1]npu_done
    DMA_CFG   = 0x08   # RW:  [0]target [4]npu_mode [5]acc_clear
    SRC_LO    = 0x0C   # RW:  src_addr[31:0]
    SRC_HI    = 0x10   # RW:  src_addr[63:32]
    BURST     = 0x14   # RW:  [7:0]burst_len [23:8]total_bursts
    SEQ       = 0x18   # RW:  [15:0]seq_len
    ACC_TEST  = 0x1C   # RW:  acc test control
    DRAIN_CMD = 0x20   # W:   [4:0]row -> triggers drain
    DRAIN_STS = 0x24   # RO:  [0]drain_valid
    # PSUM[0..15] at 0x40..0x7C
    PSUM_BASE = 0x40

    # CTRL bits
    CTRL_START_DMA  = 1 << 0
    CTRL_START_NPU  = 1 << 1
    CTRL_SWAP_BANKS = 1 << 2
    CTRL_CLEAR_DONE = 1 << 3

    # DMA_CFG bits
    CFG_TARGET_WEIGHT = 1 << 0
    CFG_MODE_PRELOAD  = 1 << 4
    CFG_ACC_CLEAR     = 1 << 5

    def __init__(self, base_addr=0x43C00000, addr_range=0x80):
        self.mmio = MMIO(base_addr, addr_range)
        print(f"NPU Driver initialized @ 0x{base_addr:08X}")

    def read(self, offset):
        return self.mmio.read(offset)

    def write(self, offset, value):
        self.mmio.write(offset, value)

    def status(self):
        s = self.read(self.STATUS)
        return {"dma_done": bool(s & 1), "npu_done": bool(s & 2)}

    def poll_dma_done(self, timeout=1.0):
        t0 = time.time()
        while time.time() - t0 < timeout:
            if self.read(self.STATUS) & 1:
                return True
        raise TimeoutError("DMA did not complete")

    def poll_npu_done(self, timeout=1.0):
        t0 = time.time()
        while time.time() - t0 < timeout:
            if self.read(self.STATUS) & 2:
                return True
        raise TimeoutError("NPU did not complete")

    def dma_load(self, src_addr, burst_len=15, total_bursts=1, target_weight=False):
        """Configure and start a DMA transfer."""
        cfg = 0
        if target_weight:
            cfg |= self.CFG_TARGET_WEIGHT | self.CFG_MODE_PRELOAD
        cfg |= self.CFG_ACC_CLEAR
        self.write(self.DMA_CFG, cfg)
        self.write(self.SRC_LO, src_addr & 0xFFFFFFFF)
        self.write(self.SRC_HI, (src_addr >> 32) & 0xFFFFFFFF)
        self.write(self.BURST, (total_bursts << 8) | (burst_len & 0xFF))
        self.write(self.CTRL, self.CTRL_START_DMA)

    def swap_and_execute(self, mode_preload=False, seq_len=16):
        """Swap SRAM banks and start NPU execution."""
        self.write(self.SEQ, seq_len)
        self.write(self.CTRL, self.CTRL_SWAP_BANKS | self.CTRL_START_NPU | self.CTRL_CLEAR_DONE)

    def drain_row(self, row):
        """Drain one row from accumulator and return 16 psum values."""
        self.write(self.DRAIN_CMD, row & 0x1F)
        # Wait for drain capture (very fast, 2 clock cycles)
        for _ in range(100):
            if self.read(self.DRAIN_STS) & 1:
                break
        return [self.read(self.PSUM_BASE + lane * 4) for lane in range(16)]

    def read_psum_matrix(self):
        """Read the full 16x16 output matrix."""
        matrix = np.zeros((16, 16), dtype=np.int32)
        for row in range(16):
            psums = self.drain_row(row)
            matrix[row, :] = psums
        return matrix


npu = NPUDriver()
print("Status:", npu.status())

## 3. Run GEMM: Weight(1) × Activation(lane_index)

The local BRAM contains:
- **Weight rows** (addr 0x0000): all bytes = 1
- **Activation rows** (addr 0x2000): byte[lane] = lane (0,1,2,...,15)

Expected result: `psum[row][col] = Σ(act[lane] × weight) = Σ(lane × 1) = 0+1+2+...+15 = 120`

In [ ]:
t_start = time.perf_counter()

# ---- Step 1: DMA Weight ----
print("[1/5] DMA Weight Load...", end=" ")
npu.dma_load(src_addr=0x0000, burst_len=15, total_bursts=1, target_weight=True)
npu.poll_dma_done()
print("DONE \u2714")

# ---- Step 2: Swap + Preload ----
print("[2/5] Swap Banks + Preload...", end=" ")
npu.swap_and_execute(mode_preload=True, seq_len=16)
npu.poll_npu_done()
print("DONE \u2714")

# ---- Step 3: DMA Activation ----
print("[3/5] DMA Activation Load...", end=" ")
cfg_act = NPUDriver.CFG_ACC_CLEAR  # target=act(0), mode=execute(0)
npu.write(NPUDriver.DMA_CFG, cfg_act)
npu.dma_load(src_addr=0x2000, burst_len=15, total_bursts=1, target_weight=False)
npu.poll_dma_done()
print("DONE \u2714")

# ---- Step 4: Swap + Execute ----
print("[4/5] Swap Banks + Execute...", end=" ")
npu.swap_and_execute(seq_len=16)
npu.poll_npu_done()
print("DONE \u2714")

# ---- Step 5: Drain Results ----
print("[5/5] Draining 16\u00d716 results...", end=" ")
result = npu.read_psum_matrix()
print("DONE \u2714")

t_elapsed = (time.perf_counter() - t_start) * 1000
print(f"\nTotal time: {t_elapsed:.2f} ms (including Python overhead)")

## 4. Result Verification

In [ ]:
EXPECTED = 120

# NumPy reference: W(16x16, all 1) @ A(16x16, each row = [0,1,...,15])
W_ref = np.ones((16, 16), dtype=np.int32)
A_ref = np.tile(np.arange(16, dtype=np.int32), (16, 1))
expected_matrix = W_ref @ A_ref.T  # each element should be 120

match = np.array_equal(result, expected_matrix)
mismatches = np.sum(result != expected_matrix)

print("\u2554" + "\u2550" * 40 + "\u2557")
if match:
    print("\u2551  \u2705  GEMM RESULT: ALL 256 CELLS PASS     \u2551")
else:
    print(f"\u2551  \u274c  GEMM RESULT: {mismatches} MISMATCHES       \u2551")
print("\u255a" + "\u2550" * 40 + "\u255d")

print(f"\nExpected psum per cell: {EXPECTED}")
print(f"NPU result range: [{result.min()}, {result.max()}]")
print(f"\nSample output (row 0): {result[0, :]}")
print(f"Sample output (row 15): {result[15, :]}")

## 5. 🔥 16×16 Output Heatmap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- NPU Hardware Output ---
im0 = axes[0].imshow(result, cmap="YlOrRd", vmin=0, vmax=max(EXPECTED * 1.1, 1))
axes[0].set_title("NPU Hardware Output\n(from PYNQ AXI-Lite)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Column (output lane)")
axes[0].set_ylabel("Row (output index)")
for i in range(16):
    for j in range(16):
        axes[0].text(j, i, str(result[i, j]), ha="center", va="center", fontsize=7,
                     color="white" if result[i, j] > EXPECTED * 0.6 else "black")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# --- NumPy Reference ---
im1 = axes[1].imshow(expected_matrix, cmap="YlOrRd", vmin=0, vmax=max(EXPECTED * 1.1, 1))
axes[1].set_title("NumPy Reference\n(software golden)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Column")
for i in range(16):
    for j in range(16):
        axes[1].text(j, i, str(expected_matrix[i, j]), ha="center", va="center", fontsize=7,
                     color="white" if expected_matrix[i, j] > EXPECTED * 0.6 else "black")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# --- Difference ---
diff = result - expected_matrix
im2 = axes[2].imshow(diff, cmap="RdYlGn_r", vmin=-5, vmax=5)
axes[2].set_title("Difference\n(NPU \u2212 Reference)", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Column")
for i in range(16):
    for j in range(16):
        axes[2].text(j, i, str(diff[i, j]), ha="center", va="center", fontsize=7)
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.suptitle("SiliconToAI — 16\u00d716 NPU GEMM Verification on PYNQ-Z2",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("npu_gemm_result.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nFigure saved: npu_gemm_result.png")

## 6. Register Dump

In [ ]:
reg_names = [
    (0x00, "CTRL"),
    (0x04, "STATUS"),
    (0x08, "DMA_CFG"),
    (0x0C, "SRC_LO"),
    (0x10, "SRC_HI"),
    (0x14, "BURST"),
    (0x18, "SEQ"),
    (0x1C, "ACC_TEST"),
    (0x20, "DRAIN_CMD"),
    (0x24, "DRAIN_STS"),
]

print("\u250c" + "\u2500" * 42 + "\u2510")
print("\u2502  NPU Register Dump" + " " * 23 + "\u2502")
print("\u251c" + "\u2500" * 42 + "\u2524")
for offset, name in reg_names:
    val = npu.read(offset)
    print(f"\u2502  0x{offset:02X}  {name:<12s}  0x{val:08X}  ({val:>6d}) \u2502")
print("\u2514" + "\u2500" * 42 + "\u2518")

## 7. NPU vs NumPy Performance Comparison

Measures wall-clock time including Python MMIO overhead.  
The NPU computes 16×16 GEMM in **31 clock cycles** (~310 ns @ 100 MHz) in hardware.

In [ ]:
# --- NPU timing (full MMIO round-trip) ---
npu_times = []
for _ in range(20):
    t0 = time.perf_counter()
    # Full sequence: DMA wt → swap+preload → DMA act → swap+exec → drain
    npu.dma_load(0x0000, 15, 1, target_weight=True)
    npu.poll_dma_done()
    npu.swap_and_execute(mode_preload=True, seq_len=16)
    npu.poll_npu_done()
    npu.dma_load(0x2000, 15, 1, target_weight=False)
    npu.poll_dma_done()
    npu.swap_and_execute(seq_len=16)
    npu.poll_npu_done()
    _ = npu.read_psum_matrix()
    npu_times.append(time.perf_counter() - t0)

# --- NumPy timing ---
numpy_times = []
W = np.ones((16, 16), dtype=np.int32)
A = np.tile(np.arange(16, dtype=np.int32), (16, 1))
for _ in range(20):
    t0 = time.perf_counter()
    _ = W @ A.T
    numpy_times.append(time.perf_counter() - t0)

npu_avg_us = np.mean(npu_times) * 1e6
np_avg_us = np.mean(numpy_times) * 1e6

print(f"NPU (MMIO round-trip):  {npu_avg_us:>10.1f} \u00b5s  (HW compute: ~0.31 \u00b5s)")
print(f"NumPy (ARM Cortex-A9):  {np_avg_us:>10.1f} \u00b5s")
print(f"\nNote: NPU time is dominated by Python MMIO overhead.")
print(f"      Pure hardware latency: 31 cycles @ 100 MHz = 310 ns")
print(f"      Throughput: 16\u00d716\u00d716\u00d72 = 8,192 int8 MACs in 31 cycles")
print(f"             = {16*16*16*2 / 31 * 100:.0f} MMAC/s @ 100 MHz")

---

### 🏗️ System Summary

| Item | Value |
|---|---|
| Array Size | 16×16 weight-stationary |
| Data Type | signed int8 × int8 → int32 psum |
| Pipeline Depth | 31 cycles |
| Clock | 100 MHz |
| Peak Throughput | 51.2 GOPS (16×16×2 OPs/cycle) |
| Interface | AXI4-Lite (control) + AXI4 (DMA) |
| FPGA | xc7z020clg400-1 (PYNQ-Z2) |
| LUT | ~28% |
| FF | ~9% |
| BRAM | ~15% |